# 03. Compress와 Isolate

Compress는 필요한 사실을 짧게 보존하고, Isolate는 질문마다 전달할 근거를 분리합니다.

## 실행 환경 준비

첫 코드 셀은 노트북에서 사용할 패키지 버전을 맞춥니다. 설치가 끝나면 이어지는 셀을 순서대로 실행합니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
# model은 이후 셀에서 공통으로 사용할 LangChain 채팅 모델 객체입니다.
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
import os

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none", max_tokens=300)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 사실·조건·예외·출처를 보존합니다

길이만 줄이면 중요한 예외를 잃을 수 있습니다. 네 필드를 모두 확인한 뒤 한 줄로 압축합니다.

In [3]:
# record는 사실·조건·예외·출처를 가진 사전이고, required는 출력 순서를 고정합니다.
# 집합 차이는 빠진 키를 찾고, 함수는 네 필드를 한 줄 문자열로 반환합니다.
def compress(record: dict[str, str]) -> str:
    required = ("fact", "condition", "exception", "source")
    missing = set(required) - record.keys()
    if missing:
        raise ValueError(f"빠진 필드: {sorted(missing)}")
    return " | ".join(f"{key}: {record[key]}" for key in required)

loan = {"fact": "7권을 14일 대출", "condition": "회원 1명 기준",
        "exception": "예약이 없으면 1회 7일 연장", "source": "E01 p.3"}
compressed = compress(loan)
print(compressed)

fact: 7권을 14일 대출 | condition: 회원 1명 기준 | exception: 예약이 없으면 1회 7일 연장 | source: E01 p.3


## 3. 질문별 입력을 분리합니다

대출 질문에는 대출 근거만 넣습니다. 태그는 읽기 구조를 돕지만 보안 경계는 아닙니다.

In [4]:
# question은 답을 구할 질문이고, compressed에는 앞 셀에서 보존한 네 필드가 있습니다.
# f-string은 두 값을 구획 표시와 함께 prompt 문자열에 삽입합니다.
question = "대출 권수, 기간, 연장 조건을 알려 주세요."
prompt = f"""[근거]
{compressed}

[질문]
{question}"""
print(prompt)

[근거]
fact: 7권을 14일 대출 | condition: 회원 1명 기준 | exception: 예약이 없으면 1회 7일 연장 | source: E01 p.3

[질문]
대출 권수, 기간, 연장 조건을 알려 주세요.


## 4. 압축한 근거로 답을 생성합니다

In [5]:
# messages는 역할과 내용을 짝지은 목록이며 system 지시를 user 질문보다 먼저 둡니다.
# invoke()는 모델을 한 번 호출하고, 반환된 AIMessage에서 text와 사용량을 읽습니다.
messages = [
    ("system", "근거의 사실·조건·예외·출처를 빠뜨리지 마세요."),
    ("user", prompt),
]
response = model.invoke(messages)
# usage_metadata는 입력·출력 토큰 수를 담지만 제공되지 않을 가능성도 검사합니다.
usage = response.usage_metadata
if usage is None:
    raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
print(response.text)
print("input_tokens:", usage.get("input_tokens"))
print("output_tokens:", usage.get("output_tokens"))

회원 1명 기준으로 **7권까지 14일 대출**할 수 있습니다.  
또한 **예약이 없으면 1회에 한해 7일 연장** 가능합니다.

- **대출 권수:** 7권
- **대출 기간:** 14일
- **연장 조건:** 예약이 없는 경우에만 1회 7일 연장 가능
- **출처:** E01 p.3
input_tokens: 94
output_tokens: 98


## 정리

압축은 짧게 만드는 작업과 보존해야 할 필드를 검사하는 작업을 함께 수행합니다. 질문별 입력은 호출 전에 분리합니다.

## 실패 재현: 압축본에서 예외를 삭제합니다

연장 예외가 빠진 레코드는 일반 규칙을 예외 대상에도 적용하게 만들 수 있습니다.

In [6]:
# broken은 의도적으로 exception 키를 뺀 레코드입니다.
# compress가 누락을 발견하면 ValueError를 내며, except가 그 실패만 포착합니다.
broken = {"fact": "7권을 14일 대출", "condition": "회원 1명 기준",
          "source": "E01 p.3"}
try:
    compress(broken)
except ValueError as error:
    print("관찰한 실패:", error)
else:
    raise AssertionError("필드 누락 오류가 발생해야 합니다.")

관찰한 실패: 빠진 필드: ['exception']


## 수정: 원문의 예외 필드를 복원합니다

누락된 예외를 원문 레코드에서 가져와 다시 압축합니다.

In [7]:
# {**broken, ...}은 기존 사전을 펼친 뒤 누락된 exception 키를 추가합니다.
# fixed_compressed는 복원한 레코드의 압축 결과이며 예외 문구 보존을 검사합니다.
fixed = {**broken, "exception": loan["exception"]}
fixed_compressed = compress(fixed)
assert "예약이 없으면 1회 7일 연장" in fixed_compressed
print("수정 후 압축:", fixed_compressed)

수정 후 압축: fact: 7권을 14일 대출 | condition: 회원 1명 기준 | exception: 예약이 없으면 1회 7일 연장 | source: E01 p.3
